##MACHINE LEARNING PROJECT - BASIC CUSTOMER CHURN

##Problem Statement
Customer retention is a critical challenge in the telecommunications industry, where acquiring new 
customers is significantly more expensive than retaining existing ones. High customer churn directly 
impacts revenue, profitability, and long-term business sustainability.

##Objective
The objective of this project is to analyze customer behavior and build a machine learning model that 
predicts customer churn using historical telecom customer data. By identifying customers who are likely
to discontinue their services, the organization can implement targeted retention strategies, such as 
personalized offers, improved service plans, or proactive customer support.

This project involves:
*Understanding customer demographic, service usage, contract, and billing information
*Performing data cleaning, exploratory data analysis (EDA), and feature engineering
*Building and evaluating predictive models to classify customers as churned or retained
*Interpreting model results to identify key factors influencing customer churn

In [1]:
#Importing necessary libraries
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("tele_comm.csv")
pd.set_option('display.max_columns', None)
df

In [3]:
df.shape

There are 7043 Rows and 21 Columns

In [4]:
df.columns

"""Dataset Column Descriptions
*customerID
Unique identifier assigned to each customer; used only for identification and not for modeling.
*gender
Indicates the customer’s gender (Male or Female).
*SeniorCitizen
Binary indicator showing whether the customer is a senior citizen (1 = Yes, 0 = No).
*Partner
Specifies whether the customer has a spouse or domestic partner.
*Dependents
Indicates whether the customer has dependent family members relying on them financially.
*tenure
Number of months the customer has been with the company.
*PhoneService
Indicates whether the customer has a phone service subscription.
*MultipleLines
Specifies whether the customer has multiple phone lines.
*InternetService
Type of internet service used by the customer (DSL, Fiber optic, or No internet service).
*OnlineSecurity
Indicates whether the customer subscribes to online security services.
*OnlineBackup
Specifies whether the customer has an online backup service.
*DeviceProtection
Indicates whether the customer has device protection coverage.
*TechSupport
Specifies whether the customer receives technical support services.
*StreamingTV
Indicates whether the customer uses streaming TV services.
*StreamingMovies
Indicates whether the customer uses streaming movie services.
*Contract
Type of contract the customer has (Month-to-month, One year, or Two year).
*PaperlessBilling
Indicates whether the customer uses paperless billing.
*PaymentMethod
Describes the method used by the customer to pay bills.
*MonthlyCharges
Amount charged to the customer on a monthly basis.
*TotalCharges
Total amount charged to the customer over the entire tenure.
*Churn
Target variable indicating whether the customer has discontinued the service."""

In [5]:
df.head()

In [6]:
df.tail()

In [7]:
#Checking the data types of columns
df.dtypes

In [8]:
#Converting the datatype of 'TotalCharges' to float
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors = 'coerce')

In [9]:
df.dtypes

In [10]:
df.sample(10)

In [11]:
df.info()

There are only 11 Null Values in Total Charges

In [12]:
df.describe()

##Observation - 
The dataset shows significant variability in customer tenure and billing amounts, indicating diverse 
customer behavior patterns. Tenure and TotalCharges exhibit wide ranges and high dispersion, making
themstrong indicators of customer loyalty and lifetime value. MonthlyCharges show moderate variability, 
suggesting potential pricing sensitivity among customers. A small portion of missing TotalCharges 
valuesis observed, likely corresponding to newly onboarded customers.

In [13]:
df.select_dtypes(include='object').nunique().sort_values(ascending=False)

In [14]:
df['Contract'].value_counts()

In [15]:
df['PaymentMethod'].value_counts()

##DATA CLEANING

In [16]:
#Checking for Null values
df.isnull().sum()

In [17]:
df1 = df[df['TotalCharges'].isnull()]
df1

In [18]:
#Replacing null values in 'TotalCharges' with 0 for tenure == 0
df.loc[df['tenure'] == 0, 'TotalCharges'] = 0

In [19]:
df['TotalCharges'].isnull().sum()

##Obsveration
Missing values in TotalCharges correspond exclusively to customers with zero tenure. Since these customers have not yet been billed, TotalCharges was logically imputed as zero to preserve business consistency.

| Option                | Correct? | Reason                        |
| --------------------- | -------- | ----------------------------- |
| Drop rows             | ❌        | Removes important churn cases |
| Median impute         | ❌        | Breaks tenure–billing logic   |
| Set to 0 for tenure=0 | ✅        | Matches real-world meaning    |

In [20]:
#Checking for duplicate values
df.duplicated().sum()

There are No duplicates in Dataset

In [21]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
df[num_cols].describe()

In [22]:
#Outliers detection

In [23]:
#Outliers detection using Box pLot
import matplotlib.pyplot as plt

for col in num_cols:
    plt.figure()
    plt.boxplot(df[col].dropna())
    plt.title(f'Boxplot of {col}')
    plt.ylabel(col)
    plt.show()


In [24]:
#Outliers detection using IQR
def iqr_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound


In [25]:
for col in num_cols:
    outliers, lb, ub = iqr_outliers(df, col)
    print(f"{col}")
    print(f"Lower Bound: {lb}, Upper Bound: {ub}")
    print(f"Number of Outliers: {outliers.shape[0]}")
    print("-" * 40)


In [26]:
#Outliers detection using Scatter plot
plt.figure()
plt.scatter(df.index, df['TotalCharges'])
plt.title('Scatter Plot of TotalCharges')
plt.xlabel('Index')
plt.ylabel('TotalCharges')
plt.show()


#Observations
Outlier analysis using boxplots, scatter plots, and IQR-based thresholds did not reveal any anomalous 
values in the numerical features. Observed extreme values were consistent with valid customer behavior
and business constraints; therefore, no outlier treatment was applied.

##EXPLORATORY DATA ANALYSIS

In [27]:
target_col = df['Churn']
target_col.value_counts()

In [28]:
target_col.value_counts(normalize=True)*100

The dataset shows a significant class imbalance, with a substantially higher proportion of customers who have not churned compared to those who have.

In [29]:
#Numerical Columns in dataset
num_cols = df.select_dtypes(include=['int', 'float']).columns.tolist()
num_cols

In [30]:
#Removing customer ID column
df.drop(columns=['customerID'], inplace=True)
df.head()

In [31]:
#Category columns in a dataset
category_cols = df.select_dtypes(include=['object']).columns.tolist()
category_cols

In [32]:
#Binary category columns
binary_categorical_cols = [
    'gender',
    'SeniorCitizen',
    'Partner',
    'Dependents',
    'PhoneService',
    'PaperlessBilling',
    'Churn'
]

In [33]:
#Multi Category Columns
multi_categorical_cols = [
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaymentMethod'
]

In [34]:
import seaborn as sns
import matplotlib.pyplot as plt
import math

n_cols = 3
n_rows = math.ceil(len(binary_categorical_cols) / n_cols)

plt.figure(figsize=(n_cols*6, n_rows*4))

for i, col in enumerate(binary_categorical_cols, 1):
    plt.subplot(n_rows, n_cols, i)

    ax = sns.countplot(x=col, data=df)
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(
                f'{int(height)}',
                (p.get_x() + p.get_width() / 2., height),
                ha='center',
                va='bottom',
                fontsize=9
            )

    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

#Observations - 
* The gender distribution is nearly balanced; however, the number of male customers is slightly higher than female customers.
* A large proportion of customers are senior citizens, indicating significant representation of older users
in the dataset.
* Partner status is almost evenly distributed, though customers without partners slightly outnumber those
with partners.
* The majority of customers do not have dependents, making them the dominant group.
* Nearly all customers subscribe to phone services, showing it is a core offering.
* Paperless billing is widely adopted, with most customers opting for this billing method.

In [35]:
n_cols = 3
n_rows = math.ceil(len(multi_categorical_cols) / n_cols)

plt.figure(figsize=(n_cols*6, n_rows*4))

for i, col in enumerate(multi_categorical_cols, 1):
    plt.subplot(n_rows, n_cols, i)

    ax = sns.countplot(x=col, data=df)
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(
                f'{int(height)}',
                (p.get_x() + p.get_width() / 2., height),
                ha='center',
                va='bottom',
                fontsize=9
            )

    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

#OBSEERVATIONS
1. Most customers have phone service; however, among them, a larger proportion use single-line connections rather than multiple lines.
2. Fiber optic is the most commonly used internet service, significantly more than DSL.
3. A majority of customers do not subscribe to online security services.
4. A relatively smaller proportion of customers use online backup and device protection services.
5. Most customers do not utilize technical support services.
6. The distribution of customers using Streaming TV and Streaming Movies services is nearly identical.
7. Month-to-month contracts are the most prevalent, followed by two-year contracts, with one-year contracts being the least common.
8. Electronic check is the most frequently used payment method, followed by mailed checks, while bank transfer and credit card payments have similar usage levels.

In [36]:
sns.countplot(data=df, x='Churn')

Almost 75% people are not not churned

In [37]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
n_cols = 2    
n_rows = math.ceil(len(num_cols) / n_cols)
plt.figure(figsize=(n_cols*6, n_rows*4))
for i, col in enumerate(num_cols, 1):
    plt.subplot(n_rows, n_cols, i)
    sns.histplot(x=col, data=df)
    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

#OBSERVATIONS
1️⃣ Tenure Distribution
Customers with very low tenure (0–10 months) and very high tenure (around 70+ months) are more frequent, while customers with mid-range tenure are relatively evenly distributed, indicating moderate variability across tenure values.
2️⃣ Monthly Charges Distribution
A large number of customers are charged the minimum monthly fee (around 20), while the majority of 
customers fall within the higher monthly charge range of approximately 70 to 110.
3️⃣ Total Charges Distribution
Higher total charges are primarily associated with long-tenure customers, while low total charges are 
dominated by recently joined customers.

In [38]:
import math
import matplotlib.pyplot as plt
import seaborn as sns

n_cols = 3
n_rows = math.ceil(len(binary_categorical_cols) / n_cols)

plt.figure(figsize=(n_cols*6, n_rows*4))

for i, col in enumerate(binary_categorical_cols, 1):
    plt.subplot(n_rows, n_cols, i)

    ax = sns.countplot(x=col, hue='Churn', data=df)
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(
                f'{int(height)}',
                (p.get_x() + p.get_width() / 2., height),
                ha='center',
                va='bottom',
                fontsize=9
            )

    plt.title(f'{col} vs Churn')
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

##OBERVATIONS
1. The proportion of churned and non-churned customers is nearly the same across male and female customers, indicating that gender does not significantly influence churn.
2. Senior citizen customers exhibit a higher churn rate compared to non–senior citizens, suggesting age
-related differences in retention behavior.
3. Customers without partners show a comparatively higher churn rate than those with partners.
4. Customers with no dependents tend to churn more than customers who have dependents.
5. Although most customers have phone services, churn is observed more frequently among phone service 
users, primarily due to their larger representation in the dataset.
6. Customers using paperless billing show a higher churn tendency compared to those using traditional 
billing methods.

In [39]:
n_cols = 3
n_rows = math.ceil(len(multi_categorical_cols) / n_cols)

plt.figure(figsize=(n_cols*6, n_rows*4))

for i, col in enumerate(multi_categorical_cols, 1):
    plt.subplot(n_rows, n_cols, i)

    ax = sns.countplot(x=col, hue='Churn', data=df)
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(
                f'{int(height)}',
                (p.get_x() + p.get_width() / 2., height),
                ha='center',
                va='bottom',
                fontsize=9
            )

    plt.title(f'{col} vs Churn')
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

##OBSERVATIONS
1. The churn percentage is nearly the same for customers with multiple lines and those without, indicating  that multiple-line service does not significantly influence churn.
2. Customers using fiber optic internet services exhibit a higher churn rate compared to those using DSL or having no internet service. ✅ (Well-known churn driver)
3. Higher churn is observed among customers who do not subscribe to add-on services such as online 
security, online backup, device protection, and technical support.
4. The churn percentages for customers using Streaming TV and Streaming Movies services are almost 
identical, suggesting similar retention behavior across these services.
5. The highest churn rates are observed among customers on month-to-month contracts and those using 
electronic check as their payment method.

In [40]:
sns.boxplot(x='Churn', y='tenure', data=df)

##observation
The boxplot indicates that churned customers have a lower median tenure and a narrower interquartile
range, while non-churned customers show higher median tenure and a broader spread, suggesting increased
retention with longer tenure.

In [41]:
sns.boxplot(x='Churn', y='MonthlyCharges', data=df)

##Observation
Customers who churn generally have higher monthly charges, primarily ranging between 60 and 100, with a 
median around 80. In contrast, non-churned customers exhibit a broader distribution of monthly charges, 
spanning approximately from 15 to 90, indicating greater variability in pricing among retained
customers.

In [42]:
#Performing mapping for churn column for heatmap
df['Churn_num'] = df['Churn'].map({'Yes':1, 'No':0})

In [43]:
num_col = df.select_dtypes(include=['int64', 'float64'])
plt.figure(figsize=(8, 6))
sns.heatmap(
    num_col.corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5
)

plt.title('Correlation Heatmap')
plt.show()

##observations - 
Tenure and TotalCharges show a strong positive correlation, indicating that customers who stay longer 
naturally accumulate higher total charges. TotalCharges and MonthlyCharges are also moderately positivelycorrelated, as higher monthly bills contribute to higher cumulative charges over time.

Tenure has a moderate negative correlation with churn, suggesting that customers with shorter tenure
are more likely to churn, while long-tenure customers tend to remain with the company.

1. MonthlyCharges has a weak positive correlation with churn, indicating that customers with higher monthly charges are slightly more prone to churn.
2. TotalCharges shows a weak negative correlation with churn, which is expected since customers who churn early do not accumulate high total charges.
3. SeniorCitizen has a weak positive correlation with churn, suggesting senior citizens churn slightly 
more, but age alone is not a strong churn driver.

In [44]:
df.groupby('Churn')['MonthlyCharges'].describe()

In [45]:
df.groupby('Contract')['Churn'].value_counts()

In [46]:
df.groupby('Contract')['Churn_num'].mean().sort_values(ascending=False)

In [47]:
df.groupby('PaymentMethod')['Churn_num'].mean().sort_values(ascending=False)

In [48]:
df.groupby('Churn')[['tenure','MonthlyCharges','TotalCharges']].mean()

In [49]:
import seaborn as sns

sns.kdeplot(data=df, x='tenure', hue='Churn', fill=True)


In [50]:
sns.pairplot(df, hue='Churn')
plt.show()

In [51]:
pd.crosstab(
    [df['Contract'], df['PaymentMethod']],
    df['Churn'],
    normalize='index'
)


##observation
Regardless of contract type, customers paying via electronic check consistently show higher churn rates,  indicating that payment method is a strong indicator of churn behavior.

In [52]:
df[['tenure','MonthlyCharges','TotalCharges','Churn_num']].corr()

In [53]:
services = [
    'OnlineSecurity','OnlineBackup','DeviceProtection',
    'TechSupport','StreamingTV','StreamingMovies'
]

for col in services:
    print(col)
    print(df.groupby(col)['Churn_num'].mean())
    print('-'*30)

##observation
Customers not utilizing value-added services consistently show a greater tendency to churn than those
who subscribe to these services, indicating the role of service engagement in customer retention.

In [54]:
df['tenure_group'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['0–12 months', '13–24 months', '25–48 months', '49–72 months']
)

pd.crosstab(df['tenure_group'], df['Churn'], normalize='index')

##observation
Customers with tenure between 0–12 months are more likely to churn compared to customers with 13–24 months of tenure, indicating that churn decreases as customer tenure increases.

In [55]:
high_risk = df[
    (df['Contract'] == 'Month-to-month') &
    (df['tenure'] < 12) &
    (df['OnlineSecurity'] == 'No') &
    (df['TechSupport'] == 'No') &
    (df['PhoneService'] == 'Yes') &
    (df['InternetService'] == 'Fiber optic')
]

In [56]:
churn_percentages = (
    high_risk['Churn']
    .value_counts(normalize=True) * 100
)

print(churn_percentages)

The cutomers who foloows this patterns or satisy this conditons are 73.54% likely to chrun

In [57]:
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

In [58]:
X = df.drop(columns=['Churn', 'Churn_num'], axis=1)
y = df['Churn']

In [59]:
X.columns

In [60]:
print(y)

In [61]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [62]:
print(X_train.shape, X_test.shape)

In [63]:
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Align columns (important)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [64]:
X_train.head()

In [65]:
X_test.shape

In [66]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [67]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

In [68]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

log_model = LogisticRegression(max_iter=1000)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [69]:
cv_scores = cross_val_score(
    log_model,
    X_train_sm,
    y_train_sm,
    cv=cv,
    scoring='roc_auc'
)

print("Cross-validation ROC-AUC:", cv_scores)
print("Mean ROC-AUC:", cv_scores.mean())

In [70]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['lbfgs']
}

In [71]:
random_search = RandomizedSearchCV(
    LogisticRegression(max_iter=1000),
    param_distributions=param_grid,
    n_iter=5,
    cv=5,
    scoring='roc_auc',
    random_state=42
)

random_search.fit(X_train_sm, y_train_sm)

print("Best Parameters:", random_search.best_params_)
print("Best CV Score:", random_search.best_score_)

In [72]:
best_model = random_search.best_estimator_

best_model.fit(X_train_sm, y_train_sm)

In [73]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

y_pred = best_model.predict(X_test_scaled)
y_prob = best_model.predict_proba(X_test_scaled)[:,1]
print("Accuracy score: ", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

print("Test ROC-AUC:", roc_auc_score(y_test, y_prob))

In [74]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.show()

In [75]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot heatmap
plt.figure(figsize=(6,5))
sns.heatmap(cm, 
            annot=True, 
            fmt='d', 
            cmap='Blues',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

In [76]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Get feature names
feature_names = X.columns

# Get coefficients
coefficients = model.coef_[0]

# Create dataframe
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

# Sort by absolute importance
feature_importance['Abs_Coefficient'] = np.abs(feature_importance['Coefficient'])
feature_importance = feature_importance.sort_values(by='Abs_Coefficient', ascending=False)

# Plot top 10 features
plt.figure(figsize=(8,6))
sns.barplot(
    x='Abs_Coefficient',
    y='Feature',
    data=feature_importance.head(10)
)

plt.title("Top 10 Important Features - Logistic Regression")
plt.xlabel("Absolute Coefficient Value")
plt.ylabel("Feature")
plt.show()
